# validation 결과 시각화

GT 박스(초록)와 예측 박스(빨강)를 같은 이미지 위에 겹쳐서 정성적으로 비교하고,
`src/evaluate.py`의 COCOeval을 이용해 mAP도 함께 확인한다.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from matplotlib.patches import Rectangle

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dataset import PillDataset
from model import build_model
from predict import run_inference
from evaluate import evaluate_map
from utils import load_config

In [ ]:
# 알약 이름이 한글이라 matplotlib 기본 폰트(DejaVu Sans)로는 네모(□)로 깨져서 나온다.
# 시스템에 설치된 한글 폰트를 찾아서 적용한다 (mac: AppleGothic/Apple SD Gothic Neo,
# Colab/Linux: Nanum Gothic 등). 아무것도 없으면 설치 안내를 출력한다.
import matplotlib.font_manager as fm

KOREAN_FONT_CANDIDATES = [
    "AppleGothic",
    "Apple SD Gothic Neo",
    "Malgun Gothic",
    "NanumGothic",
    "Nanum Gothic",
    "Noto Sans CJK KR",
    "Noto Sans KR",
]
available = {f.name for f in fm.fontManager.ttflist}
korean_font = next((name for name in KOREAN_FONT_CANDIDATES if name in available), None)

if korean_font:
    plt.rcParams["font.family"] = korean_font
    plt.rcParams["axes.unicode_minus"] = False  # 한글 폰트는 마이너스(-) 글자가 없는 경우가 많음
    print(f"한글 폰트 적용: {korean_font}")
else:
    print(
        "한글 폰트를 찾지 못했습니다. 이름이 네모(□)로 표시된다면 폰트를 설치하세요.\n"
        "  - Colab/Linux: !apt-get -y install fonts-nanum 후 런타임 재시작,\n"
        "    그 다음 matplotlib.font_manager._load_fontmanager(try_read_cache=False) 필요할 수 있음\n"
        "  - macOS: 기본 AppleGothic이 있어야 정상이라 이 메시지가 뜨면 matplotlib 폰트 캐시 문제일 수 있음"
    )

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "configs" / "fasterrcnn_test.yaml"
CHECKPOINT_PATH = PROJECT_ROOT / "outputs" / "checkpoints" / "default" / "best.pt"

config = load_config(str(CONFIG_PATH))
device = torch.device(config["train"]["device"])

val_dir = PROJECT_ROOT / "data" / "processed" / "val"
val_dataset = PillDataset(val_dir)
label_to_name = val_dataset.categories  # {label(1-index): name}

model = build_model(config)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.to(device)
model.eval()

print(f"val 이미지 {len(val_dataset)}개, 클래스 {len(label_to_name)}개")

In [ ]:
SCORE_THRESHOLD = config["predict"]["score_threshold"]
MAX_OBJECTS = config["data"]["max_objects_per_image"]


def draw_gt_vs_pred(index: int, ax=None):
    """index번째 val 이미지 위에 GT 박스(초록)와 예측 박스(빨강)를 겹쳐 그린다."""
    pil_image, gt_target = val_dataset.data[index]
    input_tensor, _ = val_dataset[index]

    boxes, labels, scores = run_inference(
        model, input_tensor, device, SCORE_THRESHOLD, MAX_OBJECTS
    )

    if ax is None:
        _, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(pil_image)
    ax.set_title(f"index={index} (image_id={gt_target['image_id'].item()})")
    ax.axis("off")

    for box, label in zip(gt_target["boxes"], gt_target["labels"]):
        x1, y1, x2, y2 = box.tolist()
        name = label_to_name.get(label.item(), str(label.item()))
        ax.add_patch(
            Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2)
        )
        ax.text(
            x1, y1, f"GT: {name}", color="black", fontsize=8,
            bbox={"facecolor": "lime", "alpha": 0.7, "pad": 1},
        )

    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box.tolist()
        name = label_to_name.get(label.item(), str(label.item()))
        ax.add_patch(
            Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linewidth=2)
        )
        ax.text(
            x1, max(y2 - 5, 0), f"{name} ({score.item():.2f})", color="white", fontsize=8,
            bbox={"facecolor": "red", "alpha": 0.7, "pad": 1},
        )

    return ax


draw_gt_vs_pred(0)
plt.show()

In [ ]:
N_COLS = 3
N_IMAGES = 9  # 앞에서부터 몇 개 볼지

n_rows = (N_IMAGES + N_COLS - 1) // N_COLS
fig, axes = plt.subplots(n_rows, N_COLS, figsize=(6 * N_COLS, 6 * n_rows))
for ax, index in zip(axes.flat, range(N_IMAGES)):
    draw_gt_vs_pred(index, ax=ax)
for ax in axes.flat[N_IMAGES:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
from ipywidgets import interact


@interact(index=(0, len(val_dataset) - 1))
def browse(index=0):
    draw_gt_vs_pred(index)
    plt.show()

## 정량 평가 (mAP)

`src/evaluate.py`의 `evaluate_map`은 val GT와 예측을 COCOeval로 비교해
mAP@[.5:.95], AP50 등을 출력한다.

In [ ]:
coco_eval = evaluate_map(model, val_dataset, device, SCORE_THRESHOLD, MAX_OBJECTS)